# Resume Matcher NLP: Empirical Evaluation & Benchmarking

This notebook evaluates:
1. **NER / Skill Extraction Performance** (Precision, Recall, F1 against ground-truth skill annotations).
2. **Rule-Based EntityRuler vs Statistical NER Baseline Comparison**.
3. **Semantic & Lexical Match Scoring** across paired resumes and job descriptions.
4. **Spearman Rank Correlation ($r_s$) & Pearson Linear Correlation ($r$)** with human ground-truth ratings.

In [ ]:
import os
import sys
import json

# Add project root to sys.path
project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.skill_extractor import SkillExtractor
from src.scorer import ResumeScorer
from eval import evaluate_skill_extraction, evaluate_matching_pairs

## 1. Skill Extraction Performance (Precision, Recall, F1)

In [ ]:
extractor = SkillExtractor()
ner_results = evaluate_skill_extraction(extractor)

print(f"Overall Precision: {ner_results['overall_precision'] * 100:.1f}%")
print(f"Overall Recall:    {ner_results['overall_recall'] * 100:.1f}%")
print(f"Overall F1-Score:  {ner_results['overall_f1'] * 100:.1f}%")

for sample_name, metrics in ner_results["samples"].items():
    print(f"\nSample: {sample_name}")
    print(f"  Precision: {metrics['precision']*100:.1f}%, Recall: {metrics['recall']*100:.1f}%, F1: {metrics['f1_score']*100:.1f}%")
    print(f"  True Positives: {metrics['true_positives']} | Missed: {metrics['missed_skills']} | Extra: {metrics['extra_skills']}")

## 2. Rule-Based vs Statistical NER Comparison

Why rule-based EntityRuler with curated taxonomy outperforms off-the-shelf statistical NER on technical resumes.

In [ ]:
benchmark_text = """
Senior ML Engineer experienced in Python, PyTorch, Scikit-Learn, and Docker on AWS.
Built RAG pipelines with LangChain and Pinecone vector search, fine-tuning BERT models.
Implemented automated CI/CD workflows and deployed microservices with FastAPI.
"""

comp = extractor.benchmark_against_statistical_ner(benchmark_text)
print("Rule-based skills detected:", comp["rule_based_count"])
print("Statistical NER detected:", comp["statistical_count"])
print("Detected by rule only:", comp["detected_by_rule_only"])
print("Analysis:", comp["analysis"])

## 3. Resume-Job Description Match Scoring & Correlation

In [ ]:
scorer = ResumeScorer(extractor)
match_results = evaluate_matching_pairs(scorer)

print(f"Spearman Rank Correlation (r_s): {match_results['spearman_rank_correlation']}")
print(f"Pearson Linear Correlation (r):   {match_results['pearson_correlation']}")
print(f"Average Target Match Score:       {match_results['avg_high_match_score'] * 100:.1f}%")
print(f"Average Mismatch Score:           {match_results['avg_low_match_score'] * 100:.1f}%")
print(f"Discrimination Margin:            +{match_results['discrimination_margin'] * 100:.1f}%")

import pandas as pd
df_pairs = pd.DataFrame(match_results["pair_results"])
df_pairs[["pair", "type", "human_ground_truth", "composite_score", "tfidf_similarity", "skill_overlap"]]